<a href="https://colab.research.google.com/github/Dgowda11/llama-spam-finetuning/blob/main/notebooks/05_base_model_baseline_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this phase we will assess how well the original, unfine-tuned Llama 3.2 3B Instruct model performs before QLoRA.

In [2]:

import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("GPU Count:", torch.cuda.device_count())

    total_memory = torch.cuda.get_device_properties(0).total_memory
    free_memory, total_memory_cuda = torch.cuda.mem_get_info()

    print(f"Total GPU Memory: {total_memory / 1024**3:.2f} GB")
    print(f"Free GPU Memory: {free_memory / 1024**3:.2f} GB")
else:
    print("GPU is not available. Check your Colab runtime settings.")


PyTorch Version: 2.14.1+cu130
CUDA Available: True
GPU Name: Tesla T4
GPU Count: 1
Total GPU Memory: 14.56 GB
Free GPU Memory: 14.46 GB


In [ ]:
# install required Dependencies

%pip install -q -U transformers accelerate datasets huggingface_hub kaggle scikit-learn pandas matplotlib seaborn


In [4]:
%pip install -q --force-reinstall pandas==2.2.3 pyarrow==23.0.1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 5.2.0 requires pyarrow>=24.0.0, but you have pyarrow 23.0.1 which is incompatible.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
numba 0.64.0 requires numpy<2.5,>=1.22, but you have numpy 2.5.5 which is incompatible.


In [41]:

# Clone the project repository

%cd /content

!git clone https://github.com/Dgowda11/sms-spam-llama-qlora.git /content/llama-spam-finetuning

%cd /content/llama-spam-finetuning

!ls


/content
Cloning into '/content/llama-spam-finetuning'...
remote: Enumerating objects: 92, done.
remote: Counting objects: 100% (92/92), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 92 (delta 40), reused 54 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (92/92), 617.18 KiB | 2.24 MiB/s, done.
Resolving deltas: 100% (40/40), done.
/content/llama-spam-finetuning
data  LICENSE  notebooks  README.md  requirements.txt  src


In [6]:

import torch
import transformers
import datasets
import pandas as pd
import sklearn

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)

print("CUDA Available:", torch.cuda.is_available())


PyTorch: 2.14.1+cu130
Transformers: 5.23.0
Datasets: 5.2.0
Pandas: 2.2.3
Scikit-learn: 1.9.1
CUDA Available: True


In [7]:
#Authenticate the Hugging Face
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")

login(token=HF_TOKEN)

print("Hugging Face authentication successful!")

Hugging Face authentication successful!


In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID
)
print("Tokenizer loaded successfully!")
print("Tokenizer:", tokenizer.__class__.__name__)
print("Vocabulary size:", tokenizer.vocab_size)
print("EOS Token:", tokenizer.eos_token)

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Tokenizer loaded successfully!
Tokenizer: TokenizersBackend
Vocabulary size: 128000
EOS Token: <|eot_id|>


Load the Base Model

In [ ]:

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

model.eval()

print("Base Llama model loaded successfully!")
print("Model dtype:", model.dtype)
print("Model device:", model.device)
print("Model training mode:", model.training)

In [10]:
# MOnitor the Resources
if torch.cuda.is_available():

    allocated = torch.cuda.memory_allocated(0) / 1024**3
    reserved = torch.cuda.memory_reserved(0) / 1024**3
    free, total = torch.cuda.mem_get_info(0)

    print(f"GPU Allocated: {allocated:.2f} GB")
    print(f"GPU Reserved: {reserved:.2f} GB")
    print(f"GPU Free: {free / 1024**3:.2f} GB")
    print(f"GPU Total: {total / 1024**3:.2f} GB")

GPU Allocated: 5.98 GB
GPU Reserved: 6.00 GB
GPU Free: 8.46 GB
GPU Total: 14.56 GB


Lets do the Zero Shot Inference

In [28]:

def predict_sms(message):

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Message: {message}"}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    prediction = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    ).strip().upper()

    # Normalize classification output
    if prediction.startswith("HAM"):
        return "ham"
    elif prediction.startswith("SPAM"):
        return "spam"
    else:
        return "unknown"


In [29]:
#Define the System Prompt
SYSTEM_PROMPT = """
You are an SMS classification assistant.

Your task is to classify SMS messages into one of two categories:
- HAM: A legitimate, normal message.
- SPAM: An unsolicited, fraudulent, promotional, or suspicious message.

Analyze the message carefully.

Respond with only one label:
HAM or SPAM.

Do not provide explanations or additional text.
"""

In [30]:
test_messages =  [
    "Congratulations! You have won a free iPhone. Click here to claim.",
    "Hey, are we still meeting for lunch today?"
]

for message in test_messages:
  prediction = predict_sms(message)

  print("SMS:", message)
  print("Prediction:", prediction)
  print("-" * 80)

SMS: Congratulations! You have won a free iPhone. Click here to claim.
Prediction: spam
--------------------------------------------------------------------------------
SMS: Hey, are we still meeting for lunch today?
Prediction: ham
--------------------------------------------------------------------------------


Execute full Test dataset inference

In [2]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/content/llama-spam-finetuning")

DATA_DIR = PROJECT_ROOT / "data" / "prepared"

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

assert TRAIN_PATH.exists(), f"Missing: {TRAIN_PATH}"
assert TEST_PATH.exists(), f"Missing: {TEST_PATH}"

print("Dataset paths verified!")

AssertionError: Missing: /content/llama-spam-finetuning/data/prepared/train.csv

In [45]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
import time
from tqdm.auto import tqdm

predictions = []
inference_times = []

model.eval()

for message in tqdm(test_df["message"], desc="Running baseline inference"):

    start_time = time.perf_counter()

    prediction = predict_sms(message)

    end_time = time.perf_counter()

    predictions.append(prediction)
    inference_times.append(end_time - start_time)

print("Inference completed!")
print("Total test samples:", len(test_df))
print("Total predictions:", len(predictions))


Running baseline inference:   0%|          | 0/1034 [00:00<?, ?it/s]

Inference completed!
Total test samples: 1034
Total predictions: 1034


Predition Evaluations

In [1]:
baseline_results = pd.DataFrame({
    "message": test_df["message"].values,
    "actual_label": test_df["label"].str.lower().values,
    "predicted_label": predictions,
    "inference_time": inference_times
})

baseline_results["correct"] = (
    baseline_results["actual_label"] ==
    baseline_results["predicted_label"]
)

display(baseline_results.head(10))

NameError: name 'pd' is not defined